# RAG Evaluation: Power BI Question Answering

This notebook builds a retrieval-augmented generation (RAG) pipeline over a Power BI ebook, answers a test set of questions, and evaluates answer quality with reference-based and model-based metrics. The final comparison ranks repeated model/configuration runs so the strongest working setup can be selected.

In [1]:
# import libraries
import pandas as pd

from langchain_ollama import OllamaLLM
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
            

/var/folders/q1/5c6nynw1077f97t4m6258zy40000gn/T/ipykernel_51116/1397412941.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/opt/anaconda3/envs/ragas_eval/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Import the RAG and evaluation tools

Load the libraries used to read the source PDF, split and embed its text, build a vector index, call a local Ollama model, and calculate evaluation metrics. The compatibility shim below supports the installed RAGAS/LangChain versions.

In [2]:
import sys
import types

# Temporary compatibility shim for RAGAS's obsolete VertexAI import
vertexai_module = types.ModuleType(
    "langchain_community.chat_models.vertexai"
)

class ChatVertexAI:
    pass

vertexai_module.ChatVertexAI = ChatVertexAI

sys.modules[
    "langchain_community.chat_models.vertexai"
] = vertexai_module

In [3]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness
)

In [4]:
pdf_reader = PyPDFLoader("Power+BI+Ebook.pdf")

In [5]:
documents = pdf_reader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7016.96it/s]


## 2. Load and prepare the source document

Read the Power BI ebook, split it into overlapping chunks, and convert each chunk into an embedding. Chunking keeps retrieved passages small enough to fit into the language model's context while overlap preserves continuity between adjacent passages.

In [6]:
db = FAISS.from_documents(chunks, embeddings)

## 3. Build the searchable vector index

Store the embedded chunks in FAISS. The retriever uses this index to find passages relevant to each question, providing the context that grounds generated answers.

In [7]:
# model call with ollama tinyllama

from langchain_ollama import ChatOllama


model = "qwen2.5:3b"
llm = ChatOllama(
    model=model,
    temperature=0,
    num_ctx=4096,
    num_predict=1024,
    format="json",
    keep_alive=0,
    client_kwargs={"timeout": 600.0}
)

llm.invoke("who are you?")


AIMessage(content='{"name":"Qwen","description":"I am a helpful and reliable language model created by Alibaba Cloud. I am here to assist you with a wide range of tasks and questions."}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-09-29T01:12:52.257247Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2814829791, 'load_duration': 42688541, 'prompt_eval_count': 33, 'prompt_eval_duration': 404461000, 'eval_count': 38, 'eval_duration': 2275588000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a0eab8-b7f9-7fd3-baf7-3b3ebd4114c3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 38, 'total_tokens': 71})

## 4. Configure the generation model

Create a local Ollama chat model with deterministic generation settings. To compare candidates, change the model name here, rerun the model setup and downstream generation/scoring cells, then give each distinct model or configuration a unique label in the final comparison cell.

In [8]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate

CQP = PromptTemplate.from_template(
"""
Given the following conversation and a follow-up question,
rephrase the follow-up question as a standalone question.

Chat History:
{chat_history}

Follow Up Input:
{question}

Standalone question:
"""
)

qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=db.as_retriever(),
    condense_question_prompt=CQP,
    return_source_documents=True,
)


## 5. Assemble the RAG question-answering chain

The chain condenses follow-up questions using chat history, retrieves relevant ebook chunks, and asks the configured model to answer from that material. Source documents are returned so retrieval evidence can also be inspected.

In [9]:
chat_history = []
query = "What are the components of Power BI tell short abouot each?"
result = qa({"question": query, "chat_history": chat_history})
print("Answer:", result["answer"])

/var/folders/q1/5c6nynw1077f97t4m6258zy40000gn/T/ipykernel_51116/1144772681.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa({"question": query, "chat_history": chat_history})


Answer: { "Power BI Relationships and KPI": "This section likely covers how to establish relationships between different datasets in Power BI and how to create Key Performance Indicators (KPIs) within the platform. Establishing relationships is crucial for creating comprehensive and integrated reports. KPIs, on the other hand, are used to measure the success of business processes and strategies, and they are often visualized in Power BI for better understanding and decision-making." }


## 6. Smoke-test one question

Run one representative question through the chain first. This confirms that document retrieval and answer generation are connected before processing the full evaluation set.

## 7. Load the evaluation questions

Read the CSV test set. Each row pairs a question with an expected answer, which is used as the reference when computing answer-quality scores.

In [10]:
#Lets add test cases for the RAG metrics
df = pd.read_csv("power_bi_rag_test_questions.csv")

In [11]:
df.head(5)

,Question,Expected Answer,Reference
0,What is Power BI?,Power BI is a business analytics service devel...,Page 4 - Introduction
1,What are the three main components of Power BI?,The three main components are Power BI Desktop...,Page 4 - Components
2,What is the common workflow of Power BI?,Connect to data sources and build a report in ...,Page 4 - Common Workflow
3,What is Power BI Desktop used for?,Power BI Desktop is a Windows application used...,Page 5 - Power BI Desktop
4,What can the Get Data feature in Power BI Desk...,It can connect to more than 80 on-premises and...,Page 5 - Get Data


In [12]:
retrieval = db.as_retriever()

In [13]:
def process_query(query):
    chat_history = []
    result = qa({"question": query, "chat_history": chat_history})
    relevant_docs = retrieval.invoke(query)
    print("Answer:", result["answer"])
    print("Relevant Documents:", relevant_docs)
    return result["answer"], relevant_docs

## 8. Define a reusable query function

For each question, call the RAG chain and retrieve the relevant passages separately. Returning both the answer and passages makes it possible to inspect what evidence was available to the model.

In [14]:
print(process_query("What are the components of Power BI tell short abouot each?"))

Answer: { "Power BI Relationships and KPI": "This section likely covers how to establish relationships between different datasets in Power BI and how to create Key Performance Indicators (KPIs) within the platform. Establishing relationships is crucial for creating comprehensive and integrated reports. KPIs, on the other hand, are used to measure the success of business processes and strategies, and they are often visualized in Power BI for better understanding and decision-making." }
Relevant Documents: [Document(id='1b6d6aeb-9046-48f5-9d5c-dff5478a467f', metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2024-03-16T19:22:04+08:00', 'author': 'Satyajit', 'moddate': '2024-03-16T19:22:04+08:00', 'source': 'Power+BI+Ebook.pdf', 'total_pages': 43, 'page': 25, 'page_label': '26'}, page_content='Copyright © 2021 \nTalentLabs Limited\nPower BI \nRelationships and \nKPI\n© zepanalytics.com26'), Document(id='888b415d-c146-4f83-804b-3

In [15]:
results = []
for index, row in df.iterrows():
    query = row['Question']
    ground_truth = row['Expected Answer']
    
    answer, relevant_docs = process_query(query)
    
    results.append({
        'user_input': query,
        'reference': ground_truth,
        'response': answer,
        'retrieved_contexts': [relevant_docs[0].page_content]
    })

Answer: { "Power BI":"is a business analytics service developed by Microsoft. It allows users to visualize and analyze data with interactive dashboards, reports, and data visualizations." }
Relevant Documents: [Document(id='2adee18b-ae54-47a6-be3d-ffa56c6def36', metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2024-03-16T19:22:04+08:00', 'author': 'Satyajit', 'moddate': '2024-03-16T19:22:04+08:00', 'source': 'Power+BI+Ebook.pdf', 'total_pages': 43, 'page': 3, 'page_label': '4'}, page_content='visualizations, as well as features for data sharing and collaboration.\n○ Power BI Mobile:\n■ This is a mobile app that allows users to view and interact with reports and\ndashboards on their mobile devices.\n■ It provides a responsive design that adapts to different screen sizes, as well\nas features for offline access and mobile-specific interactions.\nCommon Workflow of Power BI\n○ Begins by connecting to data sources and building 

## 9. Generate answers for the full test set

Loop over the CSV rows and save each question, reference answer, model response, and retrieved context. These aligned records are the shared inputs for the later evaluation methods.

In [16]:
from ragas import EvaluationDataset

In [17]:
evaluation_dataset = EvaluationDataset.from_list(results[:2])

In [18]:
from ragas.run_config import RunConfig


run_config = RunConfig(
    timeout=600,
    max_retries=1,
    max_workers=1
)

llm = ChatOllama(
    model=model,
    temperature=0,
    num_ctx=4096,
    num_predict=1024,
    format="json",
    keep_alive=0,
    client_kwargs={"timeout": 600.0}
)

evaluator_llm = LangchainLLMWrapper(
    llm,
    run_config=run_config
)

/var/folders/q1/5c6nynw1077f97t4m6258zy40000gn/T/ipykernel_51116/425020168.py:20: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


In [19]:
test_dataset = EvaluationDataset.from_list(results[:1])

## 10. Configure RAGAS evaluation

Wrap the local model as the RAGAS evaluator and set retry, timeout, and concurrency limits. The current `test_dataset` intentionally uses only the first result as a quick check; expand it to all results for a full RAGAS evaluation.

In [20]:
result = evaluate(
    dataset=test_dataset,
    metrics=[
        Faithfulness(), 
        LLMContextRecall(),
        FactualCorrectness()
    ],
    llm=evaluator_llm,
    run_config=run_config,
    raise_exceptions=True
)

result

Evaluating: 100%|██████████| 3/3 [01:56<00:00, 38.87s/it]


{'faithfulness': 0.8000, 'context_recall': 1.0000, 'factual_correctness(mode=f1)': 0.7500}

## 11. Score faithfulness, recall, and correctness

RAGAS uses the evaluator model to judge whether the answer is supported by retrieved context, whether that context covers the reference, and whether the answer is factually correct. This result is a useful diagnostic for the sampled question, separate from the multi-question score comparison below.

In [21]:
sample = results[0]

print("QUESTION:")
print(sample["user_input"])

print("\nGENERATED ANSWER:")
print(sample["response"])

print("\nREFERENCE:")
print(sample["reference"])

print("\nRETRIEVED CONTEXTS:")
for i, context in enumerate(sample["retrieved_contexts"], 1):
    print(f"\n--- Context {i} ---")
    print(context)

QUESTION:
What is Power BI?

GENERATED ANSWER:
{ "Power BI":"is a business analytics service developed by Microsoft. It allows users to visualize and analyze data with interactive dashboards, reports, and data visualizations." }

REFERENCE:
Power BI is a business analytics service developed by Microsoft that allows users to visualize and analyze data using interactive dashboards, reports, and data visualizations.

RETRIEVED CONTEXTS:

--- Context 1 ---
visualizations, as well as features for data sharing and collaboration.
○ Power BI Mobile:
■ This is a mobile app that allows users to view and interact with reports and
dashboards on their mobile devices.
■ It provides a responsive design that adapts to different screen sizes, as well
as features for offline access and mobile-specific interactions.
Common Workflow of Power BI
○ Begins by connecting to data sources and building a report in Power BI Desktop
○ Publish report from Power BI Desktop to the Power BI service
○ Share it to end u

### Inspect one evaluated example

Print the question, generated answer, reference answer, and retrieved passage together. This helps explain an evaluation score by showing both the expected response and the evidence supplied to the model.

## 12. Measure lexical overlap with BLEU and ROUGE

BLEU and ROUGE-1 compare word overlap between each generated answer and its reference. These scores are quick to interpret, but paraphrases may score lower even when their meaning is correct.

In [22]:
!pip install rouge

In [24]:
!pip install nltk
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nltk]1/2 [nltk]


In [33]:
rouge = Rouge()
blue=[]
rouge_1 = []

In [34]:
for item in results:
    reference = [item["reference"].split()]
    hypothesis = item["response"].split()
    blue.append(sentence_bleu(reference, hypothesis))
    rouge_scores = rouge.get_scores(item['response'], item["reference"])[0]
    rouge_1.append(rouge_scores['rouge-1']['f'])

In [35]:
blue

[0.5339293213731362,
 0,
 0.1607929607991094,
 0,
 0.10472900075192958,
 0,
 0,
 0,
 0.08864100090911262,
 0.4005296397635166,
 0,
 0.3698894032811369,
 0,
 0.6205718457168946,
 1.283553411781265e-78,
 0.15257340614701648,
 3.960392628134824e-155,
 4.161854235651868e-155,
 1.184298875206812e-231,
 1.324948733423365e-78]

In [36]:
rouge_1

[0.7499999950086805,
 0.0,
 0.5142857095469389,
 0.0,
 0.2816901369252133,
 0.0,
 0.0,
 0.0,
 0.35294117275017306,
 0.6666666619388889,
 0.0,
 0.7317073121475315,
 0.0,
 0.7368421002770082,
 0.37623761975492603,
 0.3265306083465223,
 0.2631578912638504,
 0.2727272678925621,
 0.24999999545000007,
 0.29411764330882356]

## 13. Measure semantic similarity with BERTScore

BERTScore compares contextual token representations rather than requiring exact word matches. The notebook records the F1 score for each answer, balancing precision and recall of semantic matches.

In [37]:
!pip install bert-score

  Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 74.8 MB/s  0:00:00
Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl (270 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [bert-score]7 [matplotlib]


In [38]:
from bert_score import score

Matplotlib is building the font cache; this may take a moment.


In [39]:
bert_scores = []

for item in results:
    reference = [item["reference"]]
    hypothesis = [item["response"]]

    P, R, F1 = score(
        hypothesis,
        reference,
        lang="en",
        verbose=False
    )

    bert_scores.append(F1.item())

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 5433.63it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 13370.46it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNE

In [40]:
print(bert_scores)

[0.9679884314537048, 0.7987473011016846, 0.9372901320457458, 0.7710556983947754, 0.9024935364723206, 0.7702874541282654, 0.8700013756752014, 0.7913634181022644, 0.8964655995368958, 0.9246108531951904, 0.792773962020874, 0.9405326843261719, 0.786665678024292, 0.9373680353164673, 0.8770444393157959, 0.8766356110572815, 0.8666720390319824, 0.85904461145401, 0.878523051738739, 0.8768309354782104]


In [41]:
import numpy as np

print("Average BERTScore:", np.mean(bert_scores))

Average BERTScore: 0.8661197423934937


## 14. Score generated text with BARTScore

BARTScore uses a pretrained sequence-to-sequence model to score how likely one text is given another. Here it scores the response against the reference; because the implementation returns negative loss, a higher (less negative) score is better.

In [42]:
!pip install transformers torch sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [43]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration

device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

model = BartForConditionalGeneration.from_pretrained(
    "facebook/bart-large-cnn"
).to(device)

model.eval()

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 3650.71it/s]


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=Tru

In [44]:
def get_bart_score(source, target):

    inputs = tokenizer(
        source,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    labels = tokenizer(
        target,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).input_ids.to(device)

    with torch.no_grad():
        output = model(
            **inputs,
            labels=labels
        )

    return -output.loss.item()

In [45]:
bart_scores = []

for item in results:

    score = get_bart_score(
        item["reference"],
        item["response"]
    )

    bart_scores.append(score)

In [46]:
print(bart_scores)

[-1.5201914310455322, -3.9291343688964844, -2.505276918411255, -7.097624778747559, -3.7295782566070557, -7.1616339683532715, -1.9470062255859375, -6.297482967376709, -2.5993399620056152, -2.4081101417541504, -4.415456771850586, -2.4567954540252686, -5.855076789855957, -1.8672517538070679, -3.537020444869995, -3.6076059341430664, -3.434171676635742, -3.3727869987487793, -3.7195026874542236, -3.495169162750244]


## 15. Compare model and generation modes

Run the answer-generation and scoring steps for each candidate using the same test questions. Update `mode_label` below to a unique description for each model/configuration; rerunning this cell updates that run's row while preserving other runs in the current kernel. The table ranks each metric independently (higher is better) and sorts by mean rank, so differently scaled metrics are not averaged directly. The first row is the best overall by this balanced rank, while the individual metric ranks show trade-offs. Restarting the kernel clears the in-memory comparison history.

In [47]:
# Change this label when comparing generation settings for the same model.
mode_label = "temperature=0 | num_ctx=4096 | num_predict=1024"
run_label = f"{llm.model} | {mode_label}"

score_columns = ["BLEU", "ROUGE-1 F1", "BERTScore F1", "BARTScore"]
current_scores = pd.DataFrame([{
    "Run": run_label,
    "BLEU": float(np.mean(blue)),
    "ROUGE-1 F1": float(np.mean(rouge_1)),
    "BERTScore F1": float(np.mean(bert_scores)),
    "BARTScore": float(np.mean(bart_scores)),
}])

if "model_comparison" not in globals():
    model_comparison = pd.DataFrame(columns=["Run", *score_columns])

model_comparison = pd.concat(
    [model_comparison, current_scores],
    ignore_index=True
).drop_duplicates(subset="Run", keep="last")

rank_columns = []
for metric in score_columns:
    rank_column = f"{metric} rank"
    model_comparison[rank_column] = model_comparison[metric].rank(
        ascending=False,
        method="average"
    )
    rank_columns.append(rank_column)

model_comparison["Mean rank"] = model_comparison[rank_columns].mean(axis=1)
model_comparison = model_comparison.sort_values("Mean rank").reset_index(drop=True)
display(model_comparison)
print(f"Best overall run by mean metric rank: {model_comparison.loc[0, 'Run']}")

,Run,BLEU,ROUGE-1 F1,BERTScore F1,BARTScore,BLEU rank,ROUGE-1 F1 rank,BERTScore F1 rank,BARTScore rank,Mean rank
0,qwen2.5:3b | temperature=0 | num_ctx=4096 | nu...,0.121583,0.290845,0.86612,-3.747811,1.0,1.0,1.0,1.0,1.0


Best overall run by mean metric rank: qwen2.5:3b | temperature=0 | num_ctx=4096 | num_predict=1024


## Conclusion

This notebook built a RAG pipeline for the Power BI ebook and evaluated its answers with lexical and semantic metrics. In the current run, **qwen2.5:3b** scored BLEU **0.122**, ROUGE-1 F1 **0.291**, BERTScore F1 **0.866**, and BARTScore **-3.748**. It ranks first with a mean rank of **1.0**, but only one model/configuration has been tested, so this is a baseline rather than evidence that it is the best option. Compare additional runs on the same test questions, and expand RAGAS evaluation beyond the first question before making a final quality decision.